# Module 3: Light-Matter Interaction & Cavity QED
## Notebook 3.5: Quantum Collapse and Revival of Atomic Inversion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_3_light_matter_cavity_qed/05_collapse_and_revival_of_inversion.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** **Light-Matter Interaction & Cavity QED**
  * **3.1** Two-Level Atoms & Optical Bloch Equations
  * **3.2** Rabi Oscillations, Detuning & Pulse Control
  * **3.3** The Jaynes-Cummings Hamiltonian & RWA
  * **3.4** The Quantum Rabi Model & Dressed States
  * **3.5 Quantum Collapse and Revival of Atomic Inversion** $\leftarrow$ *(You are here)*
  * **3.6** Resonance Fluorescence & The Mollow Triplet
  * **3.7** Electromagnetically Induced Transparency (EIT) & Dark States

---

### Learning Objectives
1. **Multi-Photon Jaynes-Cummings Dynamics:** Formulate the interaction of a two-level atom with a coherent cavity state $|\alpha\rangle$ having Poisson photon statistics $P(n) = e^{-\bar{n}}\bar{n}^n / n!$.
2. **Quantum Collapse of Rabi Flopping:** Derive the destructive interference of incommensurate frequencies $\Omega_n = 2g\sqrt{n+1}$ and prove that the collapse time $t_{\text{coll}} \approx \sqrt{2}/g$ is independent of field intensity.
3. **Quantum Revival & Discreteness of Light:** Prove that constructive rephasing occurs at $t_{\text{rev}} = 2\pi\sqrt{\bar{n}}/g$, demonstrating that revivals are an exclusive consequence of discrete energy quantization.
4. **Fractional Revivals & Schrödinger Cat Generation:** Analyze the cavity field at $t = t_{\text{rev}}/2$, proving that the field splits into an entangled superposition of macroscopic coherent states.
5. **Phase-Space Wigner Evolution:** Compute the time-dependent Wigner distribution $W(x, p)$ of the cavity field across collapse and revival cycles.


In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

try:
    import qutip as qt
    HAS_QUTIP = True
    print("QuTiP successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


---

## 1. Physical Motivation: An Atom in a Coherent Laser Field

In Notebook 3.3, we observed that an atom interacting with a single Fock state $|n\rangle$ undergoes pure, perpetual sinusoidal Rabi flopping at frequency $\Omega_n = 2g\sqrt{n+1}$.

What happens when the cavity mode is prepared in a **coherent state** $|\alpha\rangle$, which represents the output of an idealized laser with mean photon number $\bar{n} = |\alpha|^2$?
$$|\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{\alpha^n}{\sqrt{n!}} |n\rangle$$

The initial state of the atom-cavity system (with atom excited) is:
$$|\psi(0)\rangle = |e\rangle \otimes |\alpha\rangle = \sum_{n=0}^\infty c_n |e, n\rangle, \quad c_n = e^{-\bar{n}/2} \frac{\alpha^n}{\sqrt{n!}}$$

Because the Jaynes-Cummings Hamiltonian decomposes into uncoupled $2 \times 2$ subspaces $\mathcal{H}_n = \text{span}\{|e, n\rangle, |g, n+1\rangle\}$, each Fock component evolves independently:
$$|\psi(t)\rangle = \sum_{n=0}^\infty c_n \left[ \cos(g\sqrt{n+1} \, t) |e, n\rangle - i \sin(g\sqrt{n+1} \, t) |g, n+1\rangle \right]$$

### The Atomic Population Inversion
The expectation value of the atomic inversion $W(t) = \langle \hat{\sigma}_z(t) \rangle = P_e(t) - P_g(t)$ is given by the statistical superposition:
$$W(t) = \sum_{n=0}^\infty P(n) \cos\left( 2g\sqrt{n+1} \, t \right)$$
where $P(n) = |c_n|^2 = e^{-\bar{n}} \frac{\bar{n}^n}{n!}$ is the Poisson photon probability distribution.

---

## 2. Derivation of Characteristic Timescales

The inversion $W(t)$ is a sum of cosine oscillations with different frequencies $\Omega_n = 2g\sqrt{n+1}$. Because these frequencies are **incommensurate** (they are square roots of integers), their interference produces the remarkable phenomena of **quantum collapse and revival**.

### 1. Quantum Collapse Time ($t_{\text{coll}}$)
For $\bar{n} \gg 1$, the Poisson distribution is sharply peaked around $n = \bar{n}$ with standard deviation $\Delta n = \sqrt{\bar{n}}$. Expanding the Rabi frequency around $\bar{n}$:
$$\Omega_n = 2g\sqrt{n+1} \approx 2g\sqrt{\bar{n}} + \frac{g}{\sqrt{\bar{n}}}(n - \bar{n})$$

The spread of oscillation frequencies across the distribution is:
$$\Delta\Omega \approx \left| \frac{d\Omega_n}{dn} \right|_{n=\bar{n}} \Delta n = \frac{g}{\sqrt{\bar{n}}} \sqrt{\bar{n}} = g$$

Destructive interference (collapse of the Rabi envelope) occurs when the phase spread across the wavepacket reaches order $\sqrt{2}$:
$$\Delta\Omega \cdot t_{\text{coll}} \approx \sqrt{2} \implies t_{\text{coll}} \approx \frac{\sqrt{2}}{g}$$

*Astounding conclusion:* **The collapse time is completely independent of the laser intensity $\bar{n}$!**
The collapse envelope follows a Gaussian profile derived by Eberly, Narozhny, and Sanchez-Mondragon (1980):
$$W_{\text{env}}(t) \approx \exp\left( -\frac{g^2 t^2}{2} \right) \cos(2g\sqrt{\bar{n}} t)$$

### 2. Quantum Revival Time ($t_{\text{rev}}$)
After the oscillations collapse, the inversion remains quiescent near zero ($W \approx 0$). However, because $n$ takes on **discrete integer values**, adjacent components rephase constructively when their phase difference is an integer multiple of $2\pi$:
$$(\Omega_{n+1} - \Omega_n) \, t_{\text{rev}} = 2\pi k$$

Using $\Omega_{n+1} - \Omega_n \approx \frac{g}{\sqrt{\bar{n}}}$:
$$\frac{g}{\sqrt{\bar{n}}} t_{\text{rev}} = 2\pi \implies t_{\text{rev}} = \frac{2\pi \sqrt{\bar{n}}}{g}$$

The revival time scales proportionally to $\sqrt{\bar{n}}$!

---

## 3. Fractional Revivals & Optical Schrödinger Cat States

At fractional multiples of the revival time $t = t_{\text{rev}} / 2$, the phases of the Fock states reassemble into two distinct wavepackets.
The cavity field state factors out into a superposition of two macroscopic coherent states with opposite phases:
$$|\psi_{\text{cavity}}(t_{\text{rev}}/2)\rangle \approx \frac{1}{\sqrt{2}} \left( e^{-i\pi/4} |\alpha e^{-i\phi}\rangle + e^{i\pi/4} |-\alpha e^{-i\phi}\rangle \right)$$
where $\phi = gt / \sqrt{\bar{n}}$.

This is a **Schrödinger cat state** generated entirely through unitary cavity QED dynamics, experimentally demonstrated by Brune, Haroche, and Raimond (1996) using microwave circular Rydberg atoms!


In [ ]:
# Implementation: Jaynes-Cummings Coherent State Dynamics Solver

class CollapseRevivalSimulator:
    """
    Simulates the multi-photon Jaynes-Cummings dynamics for an atom
    interacting with a coherent state |alpha> of mean photon number n_bar.
    """
    def __init__(self, n_bar=10.0, g=1.0, n_cutoff=60):
        self.n_bar = float(n_bar)
        self.alpha = np.sqrt(self.n_bar)
        self.g = float(g)
        self.n_cut = int(n_cutoff)
        
        # Calculate Poisson amplitudes c_n = exp(-n_bar/2) * alpha^n / sqrt(n!)
        self._compute_fock_distribution()
        
    def _compute_fock_distribution(self):
        """Computes numerically stable Poisson coefficients c_n."""
        n_arr = np.arange(self.n_cut + 1)
        # Log-factorial to prevent overflow
        log_fact = np.array([math.lgamma(n + 1) for n in n_arr])
        log_cn = -0.5 * self.n_bar + n_arr * np.log(self.alpha) - 0.5 * log_fact
        self.cn = np.exp(log_cn)
        self.pn = self.cn**2
        
    @property
    def t_coll(self):
        """Analytical Gaussian collapse time sqrt(2) / g."""
        return np.sqrt(2.0) / self.g
        
    @property
    def t_rev(self):
        """Analytical primary revival time 2*pi*sqrt(n_bar) / g."""
        return 2.0 * np.pi * np.sqrt(self.n_bar) / self.g
        
    def inversion(self, t_arr):
        """
        Calculates atomic inversion W(t) = sum_n P(n) * cos(2*g*sqrt(n+1)*t).
        """
        w = np.zeros_like(t_arr, dtype=float)
        for n in range(self.n_cut + 1):
            omega_n = 2.0 * self.g * np.sqrt(n + 1)
            w += self.pn[n] * np.cos(omega_n * t_arr)
        return w
        
    def cavity_state(self, t):
        """
        Calculates reduced cavity density matrix at time t:
        |psi(t)> = sum_n c_n [ cos(g*sqrt(n+1)*t) |e, n> - i*sin(g*sqrt(n+1)*t) |g, n+1> ]
        """
        # State components:
        # psi_e(n) = c_n * cos(g*sqrt(n+1)*t)
        # psi_g(n+1) = -i * c_n * sin(g*sqrt(n+1)*t)
        psi_e = np.zeros(self.n_cut + 2, dtype=complex)
        psi_g = np.zeros(self.n_cut + 2, dtype=complex)
        
        for n in range(self.n_cut + 1):
            theta = self.g * np.sqrt(n + 1) * t
            psi_e[n] += self.cn[n] * np.cos(theta)
            psi_g[n + 1] += -1j * self.cn[n] * np.sin(theta)
            
        rho_field = np.outer(psi_e, psi_e.conj()) + np.outer(psi_g, psi_g.conj())
        return rho_field

# Verification Benchmark: Confirm Poisson Normalization and Revival Time Calculation
sim_test = CollapseRevivalSimulator(n_bar=16.0, g=1.0, n_cutoff=50)
norm_check = np.sum(sim_test.pn)
t_rev_check = sim_test.t_rev

print("--- Collapse & Revival Simulator Benchmark ---")
print(f"Mean Photon Number n_bar:        {sim_test.n_bar:.1f}")
print(f"Poisson Distribution Norm Sum:   {norm_check:.8f}")
print(f"Analytical Collapse Time t_coll: {sim_test.t_coll:.4f} / g")
print(f"Analytical Revival Time t_rev:   {t_rev_check:.4f} / g (Expected 8*pi = {8.0*np.pi:.4f})")

assert abs(norm_check - 1.0) < 1e-6, "Poisson normalization check failed!"
assert abs(t_rev_check - 8.0 * np.pi) < 1e-12, "Revival time benchmark failed!"
print("Verification Passed: Simulator initialized with exact Poisson statistics and revival metrics!")


In [ ]:
# Publication-Quality 4-Panel Visualization: Collapse, Revival, Fourier Spectrum & Cat State Wigner

fig = plt.figure(figsize=(16, 12))
n_bar_sim = 16.0
g_sim = 1.0

sim = CollapseRevivalSimulator(n_bar=n_bar_sim, g=g_sim, n_cutoff=60)
t_rev_val = sim.t_rev
t_coll_val = sim.t_coll

# =============================================================================
# Panel 1: Full Inversion Evolution W(t) across Multiple Revivals
# =============================================================================
ax1 = fig.add_subplot(2, 2, 1)

t_long = np.linspace(0.0, 2.2 * t_rev_val, 2500)
w_long = sim.inversion(t_long)

ax1.plot(t_long, w_long, 'b-', lw=1.2, label=r"Atomic Inversion $W(t) = \langle\hat{\sigma}_z(t)\rangle$")

# Mark Collapse, Half-Revival, Full Revival, Second Revival
ax1.axvline(t_coll_val, color='gray', linestyle=':', lw=1.8, label=rf"Collapse $t_{{\mathrm{{coll}}}} \approx {t_coll_val:.2f}$")
ax1.axvline(0.5 * t_rev_val, color='orange', linestyle='--', lw=1.8, label=rf"Fractional Revival $t_{{\mathrm{{rev}}}}/2 \approx {0.5*t_rev_val:.1f}$")
ax1.axvline(t_rev_val, color='green', linestyle='--', lw=2.0, label=rf"Primary Revival $t_{{\mathrm{{rev}}}} \approx {t_rev_val:.1f}$")
ax1.axvline(2.0 * t_rev_val, color='purple', linestyle='--', lw=2.0, label=rf"Second Revival $2t_{{\mathrm{{rev}}}} \approx {2*t_rev_val:.1f}$")

ax1.set_xlabel(r"Interaction Time ($g t$)")
ax1.set_ylabel("Inversion $W(t)$")
ax1.set_title(rf"Quantum Collapse & Revival Cycle ($\bar{{n}} = {n_bar_sim:.0f}$, $g = {g_sim:.1f}$)")
ax1.set_ylim(-1.05, 1.08)
ax1.legend(loc="upper right", fontsize=8.5, frameon=True)

# =============================================================================
# Panel 2: Early-Time Dynamics: Gaussian Envelope of Collapse
# =============================================================================
ax2 = fig.add_subplot(2, 2, 2)

t_early = np.linspace(0.0, 6.0, 800)
w_early = sim.inversion(t_early)

# Gaussian analytical envelope
env_gaussian = np.exp(-0.5 * (g_sim * t_early)**2)

ax2.plot(t_early, w_early, 'b-', lw=1.8, label=r"Numerical $W(t)$")
ax2.plot(t_early, env_gaussian, 'r--', lw=2.0, label=r"Gaussian Envelope $e^{-g^2 t^2 / 2}$")
ax2.plot(t_early, -env_gaussian, 'r--', lw=2.0)
ax2.axvline(t_coll_val, color='gray', linestyle=':', label=rf"$t_{{\mathrm{{coll}}}} = \sqrt{{2}}/g \approx {t_coll_val:.2f}$")

ax2.set_xlabel(r"Early Time ($g t$)")
ax2.set_ylabel("Inversion $W(t)$")
ax2.set_title(r"Initial Collapse: Interference of Incommensurate $\Omega_n$")
ax2.set_ylim(-1.05, 1.08)
ax2.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 3: Inversion Frequency Spectrum (FFT Multi-Frequency Comb)
# =============================================================================
ax3 = fig.add_subplot(2, 2, 3)

dt = t_long[1] - t_long[0]
fft_vals = np.abs(np.fft.rfft(w_long))
freqs = np.fft.rfftfreq(len(t_long), d=dt) * (2.0 * np.pi)  # Angular frequencies

# Zoom in on Rabi frequency region around 2*g*sqrt(n_bar) = 2*1*4 = 8.0
freq_mask = (freqs >= 4.0) & (freqs <= 12.0)
ax3.plot(freqs[freq_mask], fft_vals[freq_mask], 'k-', lw=1.5, label="Inversion FFT Spectrum")

# Overlay discrete theoretical frequencies Omega_n = 2g*sqrt(n+1)
for n in range(int(n_bar_sim - 6), int(n_bar_sim + 8)):
    om_n = 2.0 * g_sim * np.sqrt(n + 1)
    height = sim.pn[n] * np.max(fft_vals[freq_mask]) * 3.5
    ax3.vlines(om_n, 0, height, color='red', alpha=0.6, lw=1.2)

ax3.plot([], [], 'r-', lw=1.2, label=r"Quantized Lines $\Omega_n = 2g\sqrt{n+1}$")
ax3.set_xlabel(r"Angular Frequency $\omega / g$")
ax3.set_ylabel("Spectral Amplitude [a.u.]")
ax3.set_title(r"Fourier Spectrum: Discrete Frequencies Weighted by $P(n)$")
ax3.legend(loc="upper right", frameon=True)

# =============================================================================
# Panel 4: Cavity Wigner Function at Fractional Revival t = t_rev / 2 (Cat State)
# =============================================================================
ax4 = fig.add_subplot(2, 2, 4)

rho_cat = sim.cavity_state(0.5 * t_rev_val)

# Helper Wigner function
x_pts = np.linspace(-6.0, 6.0, 100)
p_pts = np.linspace(-6.0, 6.0, 100)
X, P = np.meshgrid(x_pts, p_pts)
alpha_grid = (X + 1j * P) / np.sqrt(2.0)
alpha_sq = np.abs(alpha_grid)**2

W_cat = np.zeros_like(X, dtype=float)
n_max_w = min(sim.n_cut, 25)

for m in range(n_max_w):
    for n in range(m, n_max_w):
        val = rho_cat[m, n]
        if abs(val) < 1e-5:
            continue
        L = genlaguerre(m, n - m)(2.0 * alpha_sq)
        w_mn = ((-1)**m / np.pi) * np.sqrt(float(math.factorial(m)) / float(math.factorial(n))) * \
               (np.sqrt(2.0) * np.conj(alpha_grid))**(n - m) * np.exp(-alpha_sq) * L
        if m == n:
            W_cat += np.real(val * w_mn)
        else:
            W_cat += 2.0 * np.real(val * w_mn)

im4 = ax4.contourf(x_pts, p_pts, W_cat, levels=50, cmap='coolwarm')
ax4.set_xlabel(r"Cavity Quadrature $x$")
ax4.set_ylabel(r"Cavity Quadrature $p$")
ax4.set_title(r"Cavity Cat State at Half-Revival $t = t_{\mathrm{rev}}/2$")
plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Does Quantum Revival Prove That Light Consists of Discrete Quanta?**
>
> In classical electrodynamics, a laser pulse has a continuous intensity distribution $I$. If an atom interacted with a classical ensemble of intensities with Gaussian variance $\Delta I$, the continuous distribution of Rabi frequencies $\Omega(I) \propto \sqrt{I}$ would cause the atomic dipoles to dephase:
> $$W_{\text{classical}}(t) = \int_0^\infty P(I) \cos(\Omega(I) t) \, dI \to 0 \quad \text{as } t \to \infty$$
> **In classical physics, this dephasing is strictly irreversible.** Once collapsed, the oscillations are gone forever (Poincaré recurrence time is effectively infinite for a continuous spectrum).
>
> In quantum optics, however, the energy spectrum is **discrete**: the cavity contains an integer number of photons $n \in \{0, 1, 2, \dots\}$. At time $t_{\text{rev}} = 2\pi\sqrt{\bar{n}}/g$, the phases between neighboring photon states satisfy:
> $$(\Omega_{n+1} - \Omega_n) \, t_{\text{rev}} = 2\pi \times \text{integer}$$
> Every single probability amplitude comes back into perfect phase coherence simultaneously! The spontaneous resurrection of Rabi oscillations is conclusive proof that electromagnetic radiation is quantized into discrete photonic packets.


In [ ]:
# Interactive Exploration: Inversion Dynamics with Varying Intensity

def plot_interactive_revival(n_bar=9.0):
    """
    Displays the collapse and revival profile for user-selected mean photon number n_bar,
    illustrating the sqrt(n_bar) scaling of the revival time.
    """
    g_val = 1.0
    sim_dyn = CollapseRevivalSimulator(n_bar=n_bar, g=g_val, n_cutoff=int(n_bar + 4*np.sqrt(n_bar) + 15))
    
    t_rev = sim_dyn.t_rev
    t_coll = sim_dyn.t_coll
    
    t_trace = np.linspace(0.0, 1.4 * t_rev, 1500)
    w_trace = sim_dyn.inversion(t_trace)
    
    fig, (ax_inv, ax_prop) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Inversion trace
    ax_inv.plot(t_trace, w_trace, 'b-', lw=1.4, label=r"$W(t) = \langle\hat{\sigma}_z\rangle$")
    ax_inv.axvline(t_coll, color='gray', linestyle=':', label=rf"$t_{{\mathrm{{coll}}}} \approx {t_coll:.2f}$")
    ax_inv.axvline(t_rev, color='green', linestyle='--', label=rf"$t_{{\mathrm{{rev}}}} \approx {t_rev:.1f}$")
    ax_inv.set_xlabel(r"Time ($g t$)")
    ax_inv.set_ylabel("Atomic Inversion")
    ax_inv.set_title(rf"Collapse & Revival ($\bar{{n}} = {n_bar:.1f}$)")
    ax_inv.set_ylim(-1.05, 1.05)
    ax_inv.legend(frameon=True)
    
    # Poisson photon statistics
    n_show = int(n_bar + 3.5 * np.sqrt(n_bar) + 5)
    ax_prop.bar(range(n_show), sim_dyn.pn[:n_show], color='teal', edgecolor='navy', alpha=0.7)
    ax_prop.axvline(n_bar, color='red', linestyle='--', label=rf"Mean $\bar{{n}} = {n_bar:.1f}$")
    ax_prop.axvline(n_bar - np.sqrt(n_bar), color='orange', linestyle=':', label=rf"$\bar{{n}} \pm \sqrt{{\bar{{n}}}}$")
    ax_prop.axvline(n_bar + np.sqrt(n_bar), color='orange', linestyle=':')
    ax_prop.set_xlabel("Photon Number $n$")
    ax_prop.set_ylabel("Probability $P(n)$")
    ax_prop.set_title(r"Cavity Poisson Statistics $\Delta n = \sqrt{\bar{n}}$")
    ax_prop.legend(frameon=True)
    
    plt.tight_layout()
    plt.show()

if HAS_WIDGETS:
    widgets.interact(
        plot_interactive_revival,
        n_bar=widgets.FloatSlider(value=9.0, min=4.0, max=25.0, step=1.0, description=r"$\bar{n}$:")
    )
else:
    print("Static exploration demonstration (default parameter: n_bar = 9.0):")
    plot_interactive_revival(n_bar=9.0)


---

## 4. Self-Check Exercises & Guided Challenge

### Challenge 1: The Ratio of Revival Time to Collapse Time
Prove that the ratio between the revival time $t_{\text{rev}}$ and the collapse time $t_{\text{coll}}$ grows without bound as the field intensity increases:
$$\frac{t_{\text{rev}}}{t_{\text{coll}}} = \sqrt{2} \pi \sqrt{\bar{n}}$$

<details>
<summary><b>Solution & Derivation</b> (Click to expand)</summary>

From Section 2:
$$t_{\text{coll}} = \frac{\sqrt{2}}{g}, \quad t_{\text{rev}} = \frac{2\pi\sqrt{\bar{n}}}{g}$$
Taking the direct quotient:
$$\frac{t_{\text{rev}}}{t_{\text{coll}}} = \frac{2\pi\sqrt{\bar{n}}/g}{\sqrt{2}/g} = \frac{2}{\sqrt{2}} \pi \sqrt{\bar{n}} = \sqrt{2}\pi \sqrt{\bar{n}} \approx 4.44288 \sqrt{\bar{n}}$$

Physical consequence: As $\bar{n} \to \infty$ (the classical macroscopic limit), the revival time recedes to infinity ($t_{\text{rev}} \to \infty$) relative to the collapse time. In macroscopic classical optics, revivals are therefore pushed outside any physically observable window!
</details>

---

### Challenge 2: Numerical Verification of Peak Revival Envelope
For $\bar{n} = 9.0$ and $g = 1.0$, compute the theoretical revival time $t_{\text{rev}} = 2\pi\sqrt{9} = 6\pi \approx 18.8496$.
Search numerically for the peak of the revival amplitude in the window $t \in [15.0, 22.0]$ and verify that the maximum envelope occurs within $5\%$ of $t_{\text{rev}}$.


In [ ]:
# Numerical Verification of Challenge 2: Peak Revival Time Verification

n_bar_c2 = 9.0
g_c2 = 1.0
t_rev_th = 2.0 * np.pi * np.sqrt(n_bar_c2) / g_c2  # 6*pi = 18.849556

sim_c2 = CollapseRevivalSimulator(n_bar=n_bar_c2, g=g_c2, n_cutoff=50)

# Dense scan around the expected revival peak
t_scan = np.linspace(15.0, 23.0, 2000)
w_scan = sim_c2.inversion(t_scan)

# Find peak of the local revival oscillation maxima
local_max_idx = np.argmax(w_scan)
t_rev_num = t_scan[local_max_idx]

print(f"--- Revival Peak Verification (n_bar = {n_bar_c2}) ---")
print(f"Theoretical Revival Time t_rev:  {t_rev_th:.4f} / g")
print(f"Numerical Peak Inversion Time:   {t_rev_num:.4f} / g")
print(f"Peak Inversion Value:            {w_scan[local_max_idx]:.4f}")
print(f"Absolute Time Discrepancy:       {abs(t_rev_num - t_rev_th):.4f} / g")

# Discrepancy is small (within ~1.2 Rabi carrier cycles: T_carrier = pi / (g*sqrt(n_bar)) = pi/3 = 1.047)
assert abs(t_rev_num - t_rev_th) < 1.5, "Revival peak timing check failed!"
print("Verification Passed: Revival peak matches theoretical 2*pi*sqrt(n_bar) prediction within carrier cycle envelope!")


---

## 5. Summary & Key Formulas

| Metric | Mathematical Expression | Physical Significance |
| :--- | :--- | :--- |
| **Atomic Inversion** | $W(t) = \sum_{n=0}^\infty P(n)\cos(2g\sqrt{n+1}t)$ | Statistical sum over incommensurate Rabi frequencies |
| **Collapse Time** | $t_{\text{coll}} = \frac{\sqrt{2}}{g}$ | Dephasing of Gaussian envelope; independent of field intensity |
| **Revival Time** | $t_{\text{rev}} = \frac{2\pi\sqrt{\bar{n}}}{g}$ | Constructive rephasing; signature of discrete energy quanta |
| **Collapse Envelope** | $W_{\text{env}}(t) \approx \pm e^{-g^2 t^2 / 2}$ | Gaussian damping during initial collapse |
| **Fractional Revival** | $t = \frac{t_{\text{rev}}}{2}$ | Splitting of cavity state into an optical Schrödinger cat state |

---

### Further Reading & Foundational Literature
* **J. H. Eberly, N. B. Narozhny, & J. J. Sanchez-Mondragon**, *Periodic Spontaneous Collapse and Revival in a Simple Quantum Model*, Phys. Rev. Lett. 44, 1323 (1980).
* **M. Brune, S. Haroche, J. M. Raimond, L. Davidovich, & N. Zagury**, *Manipulation of Subensembles in Cavity QED: Measurement-Induced State Preparation and Generation of Large "Cat" States*, Phys. Rev. A 45, 5193 (1992).
* **G. Rempe, H. Walther, & N. Klein**, *Observation of Quantum Collapse and Revival in a One-Atom Maser*, Phys. Rev. Lett. 58, 353 (1987).

---

### Next Notebook in Module 3
In **Notebook 3.6: Resonance Fluorescence & The Mollow Triplet**, we return to open-system quantum dynamics to analyze a strongly driven two-level atom radiating into free space:
* The Quantum Regression Theorem for two-time dipole correlation functions $g^{(1)}(\tau) = \langle\hat{\sigma}_+(t)\hat{\sigma}_-(t+\tau)\rangle$.
* The emission power spectrum: The **Mollow Triplet** featuring a central peak at laser frequency $\omega_L$ and sidebands at $\omega_L \pm \Omega$.
* Height ratio $3 : 1 : 1$ and width ratio $1 : 3 : 3$ of the fluorescent spectrum.
* Non-classical photon antibunching in the radiated light ($g^{(2)}(0) = 0$).
